In [94]:
import re
import unicodedata
import pandas as pd

In [95]:
df = pd.read_csv(r"dataset\filtering\news_balanced.csv")
print(f'''
      Shape: {df.shape}
      Columns: {df.columns.tolist()}
''')
df.head()


      Shape: (100, 6)
      Columns: ['content_id', 'text', 'section', 'published_date', 'category', 'keywords']



,content_id,text,section,published_date,category,keywords
0,1718879,With artificial intelligence (AI) in the mix a...,Tech,2025-09-29 00:00:00,5G,"5G,Internet,Technology,AI,Smart cities"
1,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17 00:00:00,Gadgets,"Gadgets,Technology"
2,1297028,"WATERTOWN, New York: A Watertown man was arres...",Tech,2024-03-04 00:00:00,Gadgets,"Gadgets,Courts Crime"
3,1782746,I knew my efforts to learn Japanese before my ...,Tech,2025-12-30 00:00:00,Gadgets,"Gadgets,Technology"
4,1345607,LONDON: British fire chiefs and recycling camp...,Tech,2024-05-13 00:00:00,Gadgets,"Gadgets,Environment"


In [96]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = text.replace('\xad', '')  # soft hyphen
    
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[-_]{2,}', ' ', text)
    text = re.sub(r'([!?.,]){2,}', r'\1', text)
    
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != "C")
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [97]:
df['clean_text'] = df['text'].apply(clean_text)

In [98]:
df['published_date'] = pd.to_datetime(df['published_date'])

In [99]:
df.head()

,content_id,text,section,published_date,category,keywords,clean_text
0,1718879,With artificial intelligence (AI) in the mix a...,Tech,2025-09-29,5G,"5G,Internet,Technology,AI,Smart cities",With artificial intelligence (AI) in the mix a...
1,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17,Gadgets,"Gadgets,Technology","CUPERTINO: Until now, the AirPods Pro were all..."
2,1297028,"WATERTOWN, New York: A Watertown man was arres...",Tech,2024-03-04,Gadgets,"Gadgets,Courts Crime","WATERTOWN, New York: A Watertown man was arres..."
3,1782746,I knew my efforts to learn Japanese before my ...,Tech,2025-12-30,Gadgets,"Gadgets,Technology",I knew my efforts to learn Japanese before my ...
4,1345607,LONDON: British fire chiefs and recycling camp...,Tech,2024-05-13,Gadgets,"Gadgets,Environment",LONDON: British fire chiefs and recycling camp...


In [100]:
df['timestamp'] = df['published_date'].astype('int64') // 10**9

In [101]:
df['year'] = df['published_date'].dt.year
df['month'] = df['published_date'].dt.month
df['week'] = df['published_date'].dt.isocalendar().week

In [102]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gaura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [103]:
df['sentences'] = df['clean_text'].apply(sent_tokenize)

In [104]:
df_sent = df.explode('sentences').reset_index(drop=True)
df_sent.rename(columns={'sentences': 'sentence'}, inplace=True)

In [105]:
category_counts = df_sent['category'].value_counts()
print(f'''{df_sent.shape}

{category_counts}
''')

(1680, 12)

AI         862
Gadgets    636
5G         182
Name: category, dtype: int64



In [106]:
df_sent = df_sent[df_sent['sentence'].str.len() > 40]

In [107]:
category_counts = df_sent['category'].value_counts()
print(f'''{df_sent.shape}

{category_counts}
''')

(1563, 12)

AI         808
Gadgets    588
5G         167
Name: category, dtype: int64

